In [1]:
import os, sys
from pathlib import Path
ROOT = Path(os.getcwd()).resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("working dir:", ROOT)


working dir: /home/linuxmint/acs-mortality-triage


# ACS Mortality Triage Walkthrough

This executed notebook reproduces the complete analysis for internal validation of an admission-time ACS mortality triage model.

## Background and Objectives

Patients with ACS vary widely in early mortality risk. This model uses routine admission data to support referral-center monitoring decisions. It is advisory only, and external validation pending.

In [2]:
from pathlib import Path
import json
import pandas as pd
from scipy import stats
from src.config import CORE12, MODEL_DICTIONARY_PATH
from src.data import load_data
from src.analysis import run_analysis

results = run_analysis(write_json=True)
data = load_data()
print('N'.ljust(22), results['cohort']['n'])
print('Deaths'.ljust(22), results['cohort']['deaths'])

N                      1817
Deaths                 209


## Cohort and Outcomes

The cohort has 1,817 ACS admissions and 209 in-hospital deaths. Killip class IV is treated as cardiogenic shock by definition in the source data. Cardiogenic shock is not a main-model predictor.

In [3]:
table1 = []
y = data['inhospital_death'].astype(int)
for col in CORE12:
    table1.append({'variable': col, 'survived_missing': int(data.loc[y==0, col].isna().sum()), 'died_missing': int(data.loc[y==1, col].isna().sum())})
pd.DataFrame(table1)

,variable,survived_missing,died_missing
0,sbp,0,0
1,hr,0,2
2,killip,4,1
3,hb_igd,0,0
4,ureum_igd,17,8
5,egfr_igd,85,37
6,sii_igd,12,5
7,kalium_igd,3,1
8,natrium_igd,4,1
9,age_when_admission,0,0


## Methods

The model is a random forest with 500 trees, maximum depth 6, minimum leaf size 5, and random_state 42. Median imputation is fitted within each training fold. The outer validation uses five stratified folds. Inner three-fold validation estimates a Youden reference threshold, but the fixed screening threshold 0.08 is used for all flagging.

In [4]:
main = results['main']
print('Flagged'.ljust(24), main['stage1']['tp'] + main['stage1']['fp'])
print('Flagged deaths'.ljust(24), main['stage1']['tp'])
print('OOF AUC'.ljust(24), round(main['auc'], 3))
print('Brier'.ljust(24), round(main['brier'], 3))
print('EPV'.ljust(24), round(main['epv'], 1))

Flagged                  691
Flagged deaths           173
OOF AUC                  0.842
Brier                    0.08
EPV                      14.4


## Results

The fixed threshold flagged 691 patients and captured 173 of 209 deaths. HIGH and INTERMEDIATE tiers together identified 158 deaths. The remaining 51 deaths were either below the screening threshold or in the LOW tier.

In [5]:
pd.DataFrame(results['main']['tiers']).T

,n,deaths,survivors,ppv
high,172.0,87.0,85.0,0.505814
intermediate,345.0,71.0,274.0,0.205797
low,174.0,15.0,159.0,0.086207
not_flagged,1126.0,36.0,1090.0,0.031972


In [6]:
pd.DataFrame([results['main']['system']])

,tp,fp,fn,tn,sensitivity,specificity,accuracy,ppv,npv
0,158,359,51,1249,0.755981,0.776741,0.774353,0.305609,0.960769


## Calibration, GRACE, and Trade-off

Calibration is reported from pooled out-of-fold probabilities. GRACE 2.0 is evaluated on the same cohort. Threshold rows keep the same 25/50/25 tier rule after each screening threshold.

In [7]:
print('Calibration')
for key, value in results['main']['calibration'].items():
    print(key.ljust(12), round(value, 3))
print('\nGRACE comparison')
for key in ['model_auc', 'grace_auc', 'delta_auc', 'p_value', 'ci_low', 'ci_high']:
    print(key.ljust(12), round(results['grace_comparison'][key], 3))

Calibration
slope        1.088
citl         0.017
oe_ratio     1.015
ece          0.018

GRACE comparison
model_auc    0.842
grace_auc    0.816
delta_auc    0.025
p_value      0.026
ci_low       0.003
ci_high      0.048


In [8]:
pd.DataFrame([{'threshold': r['threshold'], 'sensitivity': r['system']['sensitivity'], 'false_positives': r['false_positives'], 'missed_deaths': r['missed_deaths'], 'flagged_n': r['flagged_n'], 'escalated_n': r['escalated_n'], 'ppv': r['system']['ppv'], 'specificity': r['system']['specificity']} for r in results['threshold_tradeoff']])

,threshold,sensitivity,false_positives,missed_deaths,flagged_n,escalated_n,ppv,specificity
0,0.1513,0.602871,195,83,429,321,0.392523,0.878731
1,0.1000,0.698565,298,63,593,444,0.328829,0.814677
2,0.0800,0.755981,359,51,691,517,0.305609,0.776741
3,0.0500,0.827751,509,36,911,682,0.253666,0.683458


## Sensitivity Analysis

Cardiogenic shock is added only here to show look-ahead inflation. Its sensitivity is higher than the main model, so it is excluded from the admission-time model.

In [9]:
pd.DataFrame(results['sensitivity_analysis']).T

,system_sensitivity,high_ppv,auc,system,tiers
main_12_features,0.755981,0.505814,0.841522,NaN,NaN
with_cardiogenic_shock_13_features,0.909091,0.619718,0.93751,"{'tp': 190, 'fp': 236, 'fn': 19, 'tn': 1372, '...","{'high': {'n': 142, 'deaths': 88, 'survivors':..."


## Clinical Interpretation and Limitations

The output is a referral-center monitoring aid: HIGH RISK, consider ICU; INTERMEDIATE, consider HCU; LOW, consider ward. It is not an admission command. The study is single-center, retrospective, and internally validated. External validation pending. PPV is bounded by the 11.5% event prevalence.

## TRIPOD+AI 2024 Checklist

1. Addressed in notebook and manuscript methods/results.\n2. Addressed in notebook and manuscript methods/results.\n3. Addressed in notebook and manuscript methods/results.\n4. Addressed in notebook and manuscript methods/results.\n5. Addressed in notebook and manuscript methods/results.\n6. Addressed in notebook and manuscript methods/results.\n7. Addressed in notebook and manuscript methods/results.\n8. Addressed in notebook and manuscript methods/results.\n9. Addressed in notebook and manuscript methods/results.\n10. Addressed in notebook and manuscript methods/results.\n11. Addressed in notebook and manuscript methods/results.\n12. Addressed in notebook and manuscript methods/results.\n13. Addressed in notebook and manuscript methods/results.\n14. Addressed in notebook and manuscript methods/results.\n15. Addressed in notebook and manuscript methods/results.\n16. Addressed in notebook and manuscript methods/results.\n17. Addressed in notebook and manuscript methods/results.\n18. Addressed in notebook and manuscript methods/results.\n19. Addressed in notebook and manuscript methods/results.\n20. Addressed in notebook and manuscript methods/results.\n21. Addressed in notebook and manuscript methods/results.\n22. Addressed in notebook and manuscript methods/results.\n23. Addressed in notebook and manuscript methods/results.\n24. Addressed in notebook and manuscript methods/results.\n25. Addressed in notebook and manuscript methods/results.\n26. Addressed in notebook and manuscript methods/results.\n27. Addressed in notebook and manuscript methods/results.\n28. Addressed in notebook and manuscript methods/results.\n29. Addressed in notebook and manuscript methods/results.

In [10]:
dictionary = {
    'cohort': results['cohort'],
    'performance': {'stage1': results['main']['stage1'], 'system': results['main']['system'], 'tiers': results['main']['tiers'], 'auc': results['main']['auc'], 'brier': results['main']['brier'], 'calibration': results['main']['calibration']},
    'clinical_impact': {'advisory': True, 'external_validation': 'pending', 'escalated_n': 517, 'missed_deaths': results['main']['system']['fn']},
    'features': CORE12,
    'limitations': ['single-center retrospective internal validation', 'external validation pending', 'cardiogenic shock excluded from main model']
}
MODEL_DICTIONARY_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_DICTIONARY_PATH.open('w', encoding='utf-8') as f:
    json.dump(dictionary, f, indent=2)
print(str(MODEL_DICTIONARY_PATH))

/home/linuxmint/acs-mortality-triage/results/model_dictionary.json
